In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

# from huggingface_hub import snapshot_download


# # Define the target directory for inspection
# dataset_path = "/kaggle/working/jacapella"

# # Download the repository
# # We use snapshot_download to preserve the folder structure noted in your meta.csv
# snapshot_download(
#     repo_id="jaCappella/jaCappella", 
#     repo_type="dataset", 
#     local_dir=dataset_path,
#     local_dir_use_symlinks=False
# )

# print(f"Dataset downloaded to: {dataset_path}")

In [6]:
# !curl -L -o /kaggle/working/cantoria.zip "https://zenodo.org/records/5878677/files/CantoriaDataset_v1.0.0.zip?download=1"
!unzip -q /kaggle/working/cantoria.zip -d /kaggle/working/cantoria


In [12]:
!ls /kaggle/working/cantoria/Audio | head -10

Cantoria_CEA_A.wav
Cantoria_CEA_B.wav
Cantoria_CEA_MixOrgan.wav
Cantoria_CEA_Mix.wav
Cantoria_CEA_S.wav
Cantoria_CEA_T.wav
Cantoria_EJB1_A.wav
Cantoria_EJB1_B.wav
Cantoria_EJB1_MixOrgan.wav
Cantoria_EJB1_Mix.wav


In [1]:
import os
!git clone https://github.com/phos-x/MVSEP.git && cd MVSEP


# # !cat SepACap/train/configs/configs.yaml
os.chdir("/kaggle/working/MVSEP")
print(os.getcwd())

!pwd

Cloning into 'MVSEP'...
remote: Enumerating objects: 341, done.
remote: Counting objects: 100% (257/257), done.
remote: Compressing objects: 100% (161/161), done.
remote: Total 341 (delta 106), reused 236 (delta 94), pack-reused 84 (from 3)
Receiving objects: 100% (341/341), 567.22 KiB | 3.13 MiB/s, done.
Resolving deltas: 100% (129/129), done.
/kaggle/working/MVSEP
/kaggle/working/MVSEP


In [ ]:
### csd dataset processing ##### 


# import torch
# import soundfile as sf
# import numpy as np
# from pathlib import Path
# from tqdm import tqdm

# pt_folder = Path("/kaggle/working/csd")
# output_root = Path("/kaggle/working/CSD_satb")
# output_root.mkdir(parents=True, exist_ok=True)

# pt_files = sorted([f for f in pt_folder.glob("*.pt") if "split" not in f.name])
# print(f"Found {len(pt_files)} files to process.")

# def _prepare_audio_stereo(audio):
#     """
#     Standardizes audio to strictly output a NumPy array of shape [frames, 2] (Stereo).
#     This ensures soundfile.write() always creates a stereo .wav file.
#     """
#     audio = np.asarray(audio, dtype=np.float32)
    
#     # 1. Standardize orientation to [frames, channels]
#     if audio.ndim == 1:
#         # Was [frames] -> Now [frames, 1]
#         audio = np.expand_dims(audio, axis=-1) 
#     elif audio.ndim == 2:
#         # Check if it's oriented as [channels, frames] (e.g., [2, 44100])
#         if audio.shape[0] <= 4 and audio.shape[1] > audio.shape[0]:
#             audio = audio.T # Transpose to [frames, channels]

#     # 2. Enforce exactly 2 channels (Stereo)
#     channels = audio.shape[1]
#     if channels == 1:
#         # Upmix Mono to Stereo by duplicating the channel (Left and Right become identical)
#         audio = np.repeat(audio, 2, axis=1)
#     elif channels > 2:
#         # Safeguard: If audio has 3+ channels, keep only the first two
#         audio = audio[:, :2]

#     return audio

# stem_names = ["soprano", "alto", "tenor", "bass"]

# for pt_path in tqdm(pt_files, desc="Creating SATB folders"):
#     try:
#         data = torch.load(pt_path, map_location="cpu")
        
#         # mix_audio is now guaranteed to be [frames, 2]
#         mix_audio = _prepare_audio_stereo(data["mixed"].to(torch.float32).numpy())
#         src_audio = data["sources"].to(torch.float32).numpy()

#         # Handle edge case where src_audio has no stem dimension
#         if src_audio.ndim == 1:
#             src_audio = np.expand_dims(src_audio, 0)

#         track_dir = output_root / pt_path.stem
#         track_dir.mkdir(parents=True, exist_ok=True)

#         sf.write(track_dir / "mixture.wav", mix_audio, 8000)

#         for idx, name in enumerate(stem_names):
#             if idx < src_audio.shape[0]:
#                 # stem_audio is guaranteed to be [frames, 2]
#                 stem_audio = _prepare_audio_stereo(src_audio[idx])
#             else:
#                 # Because mix_audio is stereo, np.zeros_like creates a silent stereo track
#                 stem_audio = np.zeros_like(mix_audio) 
                
#             sf.write(track_dir / f"{name}.wav", stem_audio, 8000)

#     except Exception as e:
#         print(f"Failed {pt_path.name}: {e}")

# print("✅ SATB-style dataset structure created. All files are strictly stereo.")

In [ ]:
import shutil
import logging
from pathlib import Path
from typing import Dict

# 1. Setup Observability
logging.basicConfig(
    level=logging.INFO, 
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%H:%main:%S"
)

# 2. Define the Standard Map (The Data Contract)
STEM_MAP = {
    "_S.wav": "soprano.wav",
    "_A.wav": "alto.wav",
    "_T.wav": "tenor.wav",
    "_B.wav": "bass.wav",
    "_Mix.wav": "mixture.wav"
}

def is_safe_path(base_dir: Path, target_path: Path) -> bool:
    """
    Security: Prevents Directory Traversal attacks by ensuring 
    the target path is strictly a child of the base directory.
    """
    try:
        base_dir_resolved = base_dir.resolve(strict=True)
        # We don't use strict=True for target because it might not exist yet
        target_path_resolved = target_path.resolve() 
        return target_path_resolved.is_relative_to(base_dir_resolved)
    except Exception:
        return False

def group_files_by_prefix(source_dir: Path) -> Dict[str, Dict[str, Path]]:
    """
    DSA: Groups associated stems in O(N) time using a Hash Map.
    Returns: { "Cantoria_CEA": {"soprano.wav": Path(...), ...} }
    """
    grouped_files: Dict[str, Dict[str, Path]] = {}

    for file_path in source_dir.glob("*.wav"):
        filename = file_path.name
        
        # Identify if the file ends with any of our known suffixes
        matched_suffix = None
        for suffix in STEM_MAP.keys():
            if filename.endswith(suffix):
                matched_suffix = suffix
                break
        
        if matched_suffix:
            # Slice off the suffix to get the clean prefix
            prefix = filename[:-len(matched_suffix)]
            target_name = STEM_MAP[matched_suffix]
            
            if prefix not in grouped_files:
                grouped_files[prefix] = {}
            
            grouped_files[prefix][target_name] = file_path

    return grouped_files

def restructure_to_musdb(source_path: str, output_path: str) -> None:
    """
    Main orchestrator: Validates inputs, groups files, and safely copies them.
    """
    source_dir = Path(source_path)
    output_dir = Path(output_path)

    # Validate Source
    if not source_dir.exists() or not source_dir.is_dir():
        logging.error(f"Source directory '{source_dir}' is invalid or missing.")
        return

    # Ensure Output exists
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Group files efficiently
    logging.info("Scanning and grouping files...")
    grouped_files = group_files_by_prefix(source_dir)
    
    if not grouped_files:
        logging.warning("No files matching the SATB suffix map were found.")
        return

    # Execute the copy
    logging.info(f"Found {len(grouped_files)} distinct tracks. Beginning transfer...")
    
    for prefix, stems in grouped_files.items():
        track_dir = output_dir / prefix
        
        # Apply Security Sandbox
        if not is_safe_path(output_dir, track_dir):
            logging.error(f"Security Alert: Skipping malformed path '{track_dir}'")
            continue
            
        track_dir.mkdir(exist_ok=True)
        
        for target_name, source_file in stems.items():
            dest_file = track_dir / target_name
            
            # Idempotency check: Skip if file already exists to save I/O operations
            if dest_file.exists():
                continue
                
            try:
                # copy2 preserves timestamps and file metadata
                shutil.copy2(source_file, dest_file)
            except Exception as e:
                logging.error(f"Failed to copy '{source_file.name}': {e}")
                
    logging.info("Dataset restructuring complete.")

# ==========================================
# Execution Entry Point
# ==========================================
if __name__ == "__main__":
    # Change these to match your Kaggle/Local directories
    RAW_DATA_PATH = "/kaggle/working/cantoria/Audio"
    MUSDB_OUTPUT_PATH = "/kaggle/working/satb_data/test"
    
    restructure_to_musdb(RAW_DATA_PATH, MUSDB_OUTPUT_PATH)

05:04ain:31 - INFO - Scanning and grouping files...
05:04ain:31 - INFO - Found 14 distinct tracks. Beginning transfer...
05:04ain:32 - INFO - Dataset restructuring complete.


In [8]:
## cantoria augment ###

import random
import torch
import torchaudio
import torchaudio.functional as F
from pathlib import Path
from loguru import logger

class MUSDBAugmenter:
    def __init__(self, input_dir: str, output_dir: str, target_sr: int = 44100):
        # Security: Resolve absolute paths to prevent directory traversal
        self.input_dir = Path(input_dir).resolve()
        self.output_dir = Path(output_dir).resolve()
        self.target_sr = target_sr
        
        # MUSDB Strict Data Contract (Lowercase, no extensions)
        self.stems = ["soprano", "alto", "tenor", "bass"]
        
        # DSA: Pre-index the dataset into a dictionary for O(1) track lookups
        self.track_index = self._index_dataset()
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def _index_dataset(self) -> dict:
        """Scans the structured MUSDB directory and groups stems by track."""
        index = {}
        if not self.input_dir.exists():
            raise FileNotFoundError(f"Input directory not found: {self.input_dir}")
            
        for track_dir in [d for d in self.input_dir.iterdir() if d.is_dir()]:
            # Map out exactly where the 4 stems should be based on MUSDB format
            track_stems = {stem: track_dir / f"{stem}.wav" for stem in self.stems}
            
            # Only add to index if all 4 SATB stems exist (Ensures data integrity)
            if all(path.exists() for path in track_stems.values()):
                index[track_dir.name] = track_stems
                
        logger.info(f"Indexed {len(index)} complete MUSDB-formatted tracks.")
        return index

    def _load_and_resample(self, file_path: Path) -> torch.Tensor:
        """Securely loads and normalizes audio to a consistent sample rate."""
        waveform, sr = torchaudio.load(str(file_path))
        if sr != self.target_sr:
            waveform = F.resample(waveform, orig_freq=sr, new_freq=self.target_sr)
        return waveform

    def _save_stems_and_mix(self, stems: list, track_name: str, phase_name: str):
        """Mathematically sums the augmented stems and saves them to disk in MUSDB format."""
        # Create a new standard folder for this specific augmentation variation
        out_path = self.output_dir / f"{track_name}_{phase_name}"
        out_path.mkdir(parents=True, exist_ok=True)
        
        # Save individual augmented stems
        for stem_tensor, name in zip(stems, self.stems):
            torchaudio.save(str(out_path / f"{name}.wav"), stem_tensor, self.target_sr)
            
        # Mathematically reconstruct and save the mixture
        mixture = torch.stack(stems, dim=0).sum(dim=0)
        
        # Prevent digital clipping after summation
        mixture = torch.clamp(mixture, min=-1.0, max=1.0) 
        
        # Force strict lowercase MUSDB mixture format
        torchaudio.save(str(out_path / "mixture.wav"), mixture, self.target_sr)

    def generate_phase_1(self, num_variations: int = 2):
        """
        Phase 1: Supervised Augmentation. 
        Applies IDENTICAL transformations to all stems synchronously.
        """
        logger.info("Starting Phase 1: Synchronous Augmentations...")
        
        for track_name, stem_paths in self.track_index.items():
            raw_stems = [self._load_and_resample(stem_paths[name]) for name in self.stems]
            
            for var_idx in range(num_variations):
                # 1. Roll the dice ONCE per variation to ensure synchronous application
                gain_db = random.uniform(-6.0, 6.0)
                pitch_steps = random.choice([-2, -1, 0, 1, 2])
                
                # DSP Logic for EQ: Apply random coloration simulating different mics
                center_freq = random.uniform(500, 4000)
                eq_gain = random.uniform(-3.0, 3.0)
                
                augmented_stems = []
                for stem in raw_stems:
                    # Apply Gain
                    gain_linear = 10 ** (gain_db / 20.0)
                    proc_stem = stem * gain_linear
                    
                    # Apply Pitch Shift (Only if not 0 to save CPU cycles)
                    if pitch_steps != 0:
                        proc_stem = F.pitch_shift(proc_stem, self.target_sr, n_steps=pitch_steps)
                        
                    # Apply EQ Coloration
                    proc_stem = F.equalizer_biquad(proc_stem, self.target_sr, center_freq, gain=eq_gain, Q=0.707)
                    
                    augmented_stems.append(proc_stem)
                    
                self._save_stems_and_mix(augmented_stems, track_name, f"Phase1_var{var_idx}")
                
        logger.success("Phase 1 complete.")

    def generate_phase_2(self, num_synthetic_tracks: int = 10):
        """
        Phase 2: Data Expansion (Highest ROI).
        Randomly samples isolated stems across different tracks.
        """
        logger.info(f"Starting Phase 2: Generating {num_synthetic_tracks} Cross-Track Remixes...")
        track_keys = list(self.track_index.keys())
        
        for i in range(num_synthetic_tracks):
            synthetic_stems = []
            selected_sources = []
            
            for stem_name in self.stems:
                random_track = random.choice(track_keys)
                selected_sources.append(f"{stem_name[:3]}-{random_track}")
                
                stem_path = self.track_index[random_track][stem_name]
                stem_tensor = self._load_and_resample(stem_path)
                
                # Apply INDEPENDENT random gain to force robust unmixing
                ind_gain = 10 ** (random.uniform(-4.0, 4.0) / 20.0)
                synthetic_stems.append(stem_tensor * ind_gain)
                
            # DSA: Align tensor lengths to the shortest stem in the dissonant group
            min_length = min(stem.shape[1] for stem in synthetic_stems)
            aligned_stems = [stem[:, :min_length] for stem in synthetic_stems]
            
            # Generate a unique hash name for the synthetic track
            synth_name = f"SynthMix_{i}"
            self._save_stems_and_mix(aligned_stems, synth_name, "Phase2")
            
        logger.success("Phase 2 complete.")

if __name__ == "__main__":
    # --- Execution Parameters ---
    # Point input to the output of your previous restructuring script!
    INPUT_MUSDB_DIR = "/kaggle/working/satb_data/test" 
    OUTPUT_AUGMENTED_DIR = "/kaggle/working/satb_data/augment"
    
    try:
        augmenter = MUSDBAugmenter(
            input_dir=INPUT_MUSDB_DIR, 
            output_dir=OUTPUT_AUGMENTED_DIR,
            target_sr=44100 
        )
        
        # Execute the strategies
        augmenter.generate_phase_1(num_variations=2) 
        augmenter.generate_phase_2(num_synthetic_tracks=20) 
        
    except Exception as e:
        logger.error(f"Augmentation pipeline failed: {e}")

2026-04-17 12:50:03.287 | INFO     | __main__:_index_dataset:38 - Indexed 14 complete MUSDB-formatted tracks.
2026-04-17 12:50:03.289 | INFO     | __main__:generate_phase_1:72 - Starting Phase 1: Synchronous Augmentations...
2026-04-17 12:58:32.091 | SUCCESS  | __main__:generate_phase_1:103 - Phase 1 complete.
2026-04-17 12:58:32.093 | INFO     | __main__:generate_phase_2:110 - Starting Phase 2: Generating 20 Cross-Track Remixes...
2026-04-17 12:58:48.711 | SUCCESS  | __main__:generate_phase_2:136 - Phase 2 complete.


In [2]:
! ls /kaggle/working/satb_data/augment/

Cantoria_CEA_Phase1_var0   Cantoria_LNG_Phase1_var0  SynthMix_13_Phase2
Cantoria_CEA_Phase1_var1   Cantoria_LNG_Phase1_var1  SynthMix_14_Phase2
Cantoria_EJB1_Phase1_var0  Cantoria_RRC_Phase1_var0  SynthMix_15_Phase2
Cantoria_EJB1_Phase1_var1  Cantoria_RRC_Phase1_var1  SynthMix_16_Phase2
Cantoria_EJB2_Phase1_var0  Cantoria_SSS_Phase1_var0  SynthMix_17_Phase2
Cantoria_EJB2_Phase1_var1  Cantoria_SSS_Phase1_var1  SynthMix_18_Phase2
Cantoria_HCB_Phase1_var0   Cantoria_THM_Phase1_var0  SynthMix_19_Phase2
Cantoria_HCB_Phase1_var1   Cantoria_THM_Phase1_var1  SynthMix_1_Phase2
Cantoria_LBM1_Phase1_var0  Cantoria_VBP_Phase1_var0  SynthMix_2_Phase2
Cantoria_LBM1_Phase1_var1  Cantoria_VBP_Phase1_var1  SynthMix_3_Phase2
Cantoria_LBM2_Phase1_var0  Cantoria_YSM_Phase1_var0  SynthMix_4_Phase2
Cantoria_LBM2_Phase1_var1  Cantoria_YSM_Phase1_var1  SynthMix_5_Phase2
Cantoria_LJT1_Phase1_var0  SynthMix_0_Phase2	     SynthMix_6_Phase2
Cantoria_LJT1_Phase1_var1  SynthMix_10_Phase2	     SynthMix_7_Phase2
Cant

In [ ]:
val_root = Path("/kaggle/working/satb_data/test")
val_root.mkdir(parents=True, exist_ok=True)

all_dirs = sorted(
    [p for p in output_root.iterdir() if p.is_dir() and p.name not in {"validation", "test"}]
)

num_to_move = min(15, len(all_dirs))
folders_to_move = all_dirs[:num_to_move]

for folder in folders_to_move:
    dest = val_root / folder.name
    folder.rename(dest)

print(f"Moved {len(folders_to_move)} folders into {val_root}")

In [ ]:
!ls /kaggle/working/satb_test/

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.4 MB/s eta 0:00:00


In [ ]:
import yaml
import copy

# 1. Paths
original_config_path = '/kaggle/working/MVSEP/configs/config_musdb18_bs_roformer.yaml'
new_config_path = '/kaggle/working/MVSEP/configs/satb_bs_roformer.yaml'

# 2. Load the FULL original config
with open(original_config_path, 'r') as f:
    # Use FullLoader to ensure all YAML blocks (including anchors/aliases) are captured
    full_config = yaml.load(f, Loader=yaml.FullLoader)

# 3. Create a deep copy so we don't accidentally corrupt the original object
new_config = copy.deepcopy(full_config)

# 4. Modify ONLY the dataset section
new_config['training']['instruments'] = ["soprano", "alto", "tenor", "bass"]
new_config['training']['target_instrument'] = None
new_config['training']['num_epochs'] = 100
new_config['training']['num_steps'] = 100
new_config['training']['batch_size'] = 2
new_config['training']['gradient_accumulation_steps'] = 5
new_config['model']['stereo'] = "True"
new_config['model']['stft_win_length'] = 4096
new_config['model']['stft_n_fft'] = 4096
new_config['model']['dim_freqs_in'] = 2049
new_config['audio']['dim_f'] = 2048
new_config['audio']['n_fft'] = 4096
# The SATB Custom Band-Split (Must sum exactly to 2049)
new_config['model']['freqs_per_bands'] = (
    [2] * 20 +    # High Res Lows/Mids (Bass, Tenor, Alto)
    [4] * 20 +    # Mid-Highs (Soprano & Formants)
    [12] * 10 +   # Sibilance & Breath
    [24] * 10 + 
    [48] * 10 + 
    [128] * 8 +   # "Air" band (Above 10kHz)
    [65]          # Remainder to hit exactly 2049
)

# Optional: You can also wrap it in a tuple() if your config parser requires strict tuples
# new_config['model']['freqs_per_bands'] = tuple(...)

new_config['lora'] = {
    'r': 8,
    'lora_alpha': 16, # alpha / rank > 1
    'lora_dropout': 0.05,
    'merge_weights': False,
    'fan_in_fan_out': False,
    #'enable_lora': [True]
}

# 5. Save the complete config back to a new file
with open(new_config_path, 'w') as f:
    # sort_keys=False preserves the original order of the config file
    yaml.dump(new_config, f, default_flow_style=False, sort_keys=False)

print(f"✅ Fixed! The full config (including model/optimizer blocks) is now in: {new_config_path}")


✅ Fixed! The full config (including model/optimizer blocks) is now in: /kaggle/working/MVSEP/configs/satb_bs_roformer.yaml


In [ ]:
#! pip install -r requirements.txt
#%pip install loguru loralib audiomentations pedalboard auraloss torch_log_wmse rotary_embedding_torch wandb resampy --upgrade

# !cat  /kaggle/working/MVSEP/configs/satb_bs_roformer.yaml

In [7]:
!python train.py \
    --model_type bs_roformer \
    --config_path configs/satb_bs_roformer.yaml \
    --results_path results/ \
    --data_path '/kaggle/working/satb_data/augment' \
    --valid_path '/kaggle/working/satb_data/valid' \
    --num_workers 4 \
    --metrics si_sdr l1_freq sdr \
    --metric_for_scheduler sdr \
    --wandb_key "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"
    #--train_lora_loralib

GPU Compute Capability below 8.0, using math or mem efficient attention if input tensor is on cuda
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: phos-x (zahemen9900-ai) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/MVSEP/wandb/run-20260426_044040-ria0guky
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run bs_rof

In [ ]:
# import wave
# from pathlib import Path

# def duplicate_mono_to_stereo(frames: bytes, sampwidth: int) -> bytes:
#     out = bytearray(len(frames) * 2)
#     for i in range(0, len(frames), sampwidth):
#         chunk = frames[i:i+sampwidth]
#         out[i * 2:i * 2 + sampwidth] = chunk
#         out[i * 2 + sampwidth:i * 2 + sampwidth * 2] = chunk
#     return bytes(out)

# def convert_wav_to_stereo(wav_path: Path) -> bool:
#     with wave.open(str(wav_path), "rb") as src:
#         if src.getnchannels() != 1:
#             return False
#         params = src.getparams()
#         frames = src.readframes(params.nframes)

#     stereo_frames = duplicate_mono_to_stereo(frames, params.sampwidth)
#     tmp_path = wav_path.with_suffix(".tmp.wav")

#     with wave.open(str(tmp_path), "wb") as dst:
#         dst.setparams(params._replace(nchannels=2))
#         dst.writeframes(stereo_frames)

#     tmp_path.replace(wav_path)
#     return True

# root = Path("/kaggle/working/satb_test")
# converted_count = 0
# skipped_count = 0

# for wav_path in sorted(root.rglob("*.wav")):
#     try:
#         if convert_wav_to_stereo(wav_path):
#             converted_count += 1
#         else:
#             skipped_count += 1
#     except wave.Error as e:
#         print(f"Could not convert {wav_path}: {e}")

# print(f"Converted mono files to stereo: {converted_count}")
# print(f"Skipped already stereo/other channel count files: {skipped_count}")

Converted mono files to stereo: 70
Skipped already stereo/other channel count files: 0


In [ ]:
! ls -l results